# Lesson 2 — Qubits, Gates & QuantumCircuit

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 2. Work through the cells in order.

**Topics covered:**
- The qubit as a state vector
- The `QuantumCircuit` API
- Single-qubit gates: X, H, S, T, Rz
- Visualizing quantum states with `Statevector` and the Bloch sphere
- Multi-qubit gates: CX, CZ
- Circuit visualization options
- Gate identity verification with `Operator`

## Setup

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer import AerSimulator

print("All imports successful.")

## 1. The Qubit as a State Vector

A qubit state is a unit vector in a 2-dimensional complex space:
$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1$$

The `Statevector` class represents this vector exactly.

In [ ]:
# The two computational basis states
sv_0 = Statevector.from_label('0')   # |0⟩ = [1, 0]
sv_1 = Statevector.from_label('1')   # |1⟩ = [0, 1]

print("|0⟩:", sv_0)
print("|1⟩:", sv_1)

In [ ]:
# Some named single-qubit states
for label in ['0', '1', '+', '-', 'r', 'l']:
    sv = Statevector.from_label(label)
    print(f"|{label}⟩ = {np.round(sv.data, 4)}")

In [ ]:
# Build a custom normalized state
alpha = 1 / np.sqrt(2)
beta  = 1j / np.sqrt(2)   # imaginary amplitude

sv_custom = Statevector([alpha, beta])
print("State:       ", sv_custom)
print("Is valid:    ", sv_custom.is_valid())  # checks normalization
print("Probabilities:", sv_custom.probabilities_dict())

## 2. The QuantumCircuit API

In [ ]:
# Creating circuits in different ways

# Basic: n qubits, no classical bits
qc1 = QuantumCircuit(3)

# With classical bits for measurement
qc2 = QuantumCircuit(3, 3)

# With named registers
from qiskit.circuit import QuantumRegister, ClassicalRegister
qreg = QuantumRegister(2, name='q')
creg = ClassicalRegister(2, name='c')
qc3 = QuantumCircuit(qreg, creg)

print("qc1:", qc1.num_qubits, "qubits")
print("qc3 registers:", qc3.qregs, qc3.cregs)

In [ ]:
# Circuit properties
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.t(0)
qc.s(2)

print("Qubits:    ", qc.num_qubits)
print("Depth:     ", qc.depth())
print("Gates:     ", dict(qc.count_ops()))
print("Width:     ", qc.width())
qc.draw('mpl')

In [ ]:
# Barriers separate logical sections and block transpiler reordering
qc_b = QuantumCircuit(2)
qc_b.h(0)
qc_b.h(1)
qc_b.barrier()
qc_b.cx(0, 1)
qc_b.barrier()
qc_b.measure_all()
qc_b.draw('mpl')

In [ ]:
prep = QuantumCircuit(2)
prep.x(0)

entangle = QuantumCircuit(2)
entangle.h(0)
entangle.cx(0, 1)

full = prep.compose(entangle)
full.draw('mpl')

## 3. Single-Qubit Gates

For each gate we will:
1. Show its matrix with `Operator`
2. Apply it to $|0\rangle$ with `Statevector`
3. Visualize the result on the Bloch sphere

In [ ]:
# Helper: show gate matrix and its effect on |0⟩ and |1⟩
def show_gate(gate_name, qc_func):
    qc = QuantumCircuit(1)
    qc_func(qc)
    op = Operator(qc)
    print(f"--- {gate_name} ---")
    print("Matrix:")
    print(np.round(op.data, 4))
    for label in ['0', '1']:
        out = Statevector.from_label(label).evolve(qc)
        print(f"  |{label}⟩ -> {np.round(out.data, 4)}")
    print()

In [ ]:
show_gate('X (Pauli-X / NOT)', lambda qc: qc.x(0))

In [ ]:
show_gate('H (Hadamard)', lambda qc: qc.h(0))

In [ ]:
show_gate('S (Phase)', lambda qc: qc.s(0))

In [ ]:
show_gate('T', lambda qc: qc.t(0))

In [ ]:
# Rz(theta) for several angles
for theta_label, theta in [('pi/4', np.pi/4), ('pi/2', np.pi/2), ('pi', np.pi)]:
    qc = QuantumCircuit(1)
    qc.rz(theta, 0)
    op = Operator(qc)
    print(f"Rz({theta_label}):")
    print(np.round(op.data, 4))
    print()

### Bloch sphere visualization

Each row below shows a different single-qubit gate applied to $|0\rangle$. Notice where on the sphere each gate sends the state.

In [ ]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_bloch_vector
from qiskit.quantum_info import Pauli

def bloch_vector(sv):
    """Return the [x, y, z] Bloch vector for a single-qubit Statevector."""
    x = sv.expectation_value(Pauli('X')).real
    y = sv.expectation_value(Pauli('Y')).real
    z = sv.expectation_value(Pauli('Z')).real
    return [x, y, z]

gates = {
    'Identity (|0⟩)': lambda qc: None,
    'X':               lambda qc: qc.x(0),
    'H':               lambda qc: qc.h(0),
    'S':               lambda qc: qc.s(0),
    'T':               lambda qc: qc.t(0),
    'H then S':        lambda qc: (qc.h(0), qc.s(0)),
}

fig, axes = plt.subplots(1, len(gates), figsize=(18, 4),
                         subplot_kw={'projection': '3d'})

for ax, (label, apply_gate) in zip(axes, gates.items()):
    qc = QuantumCircuit(1)
    apply_gate(qc)
    sv = Statevector.from_label('0').evolve(qc)
    plot_bloch_vector(bloch_vector(sv), title=label, ax=ax)

plt.tight_layout()
plt.show()

### Gate identities

Verify these well-known identities in code:
- $H^2 = I$
- $S^2 = Z$
- $T^2 = S$
- $HXH = Z$

In [ ]:
def check_identity(name, qc_a, qc_b):
    equal = np.allclose(Operator(qc_a).data, Operator(qc_b).data)
    print(f"{name}: {'True' if equal else 'FALSE'}")

# H^2 = I
hh = QuantumCircuit(1); hh.h(0); hh.h(0)
identity = QuantumCircuit(1)  # empty circuit = identity
check_identity('H^2 = I', hh, identity)

# S^2 = Z
ss = QuantumCircuit(1); ss.s(0); ss.s(0)
z  = QuantumCircuit(1); z.z(0)
check_identity('S^2 = Z', ss, z)

# T^2 = S
tt = QuantumCircuit(1); tt.t(0); tt.t(0)
s  = QuantumCircuit(1); s.s(0)
check_identity('T^2 = S', tt, s)

# HXH = Z
hxh = QuantumCircuit(1); hxh.h(0); hxh.x(0); hxh.h(0)
check_identity('HXH = Z', hxh, z)

## 4. Multi-Qubit Gates

In [ ]:
# CX (CNOT) matrix
qc = QuantumCircuit(2)
qc.cx(0, 1)
print("CX matrix:")
print(np.round(Operator(qc).data.real).astype(int))
qc.draw('mpl')

In [ ]:
# CX action on all four basis states
qc_cx = QuantumCircuit(2)
qc_cx.cx(0, 1)

print("CX truth table (control=qubit0, target=qubit1):")
print(f"{'Input':>8}  {'Output':>8}")
for label in ['00', '01', '10', '11']:
    out = Statevector.from_label(label).evolve(qc_cx)
    # find the basis state with amplitude 1
    out_label = list(out.probabilities_dict().keys())[0]
    print(f"  |{label}⟩  ->  |{out_label}⟩")

In [ ]:
# CZ matrix
qc = QuantumCircuit(2)
qc.cz(0, 1)
print("CZ matrix:")
print(np.round(Operator(qc).data.real).astype(int))
qc.draw('mpl')

In [ ]:
# CX and CZ are interconvertible via Hadamard
# CX = (I ⊗ H) · CZ · (I ⊗ H)

cx_direct = QuantumCircuit(2)
cx_direct.cx(0, 1)

cx_from_cz = QuantumCircuit(2)
cx_from_cz.h(1)
cx_from_cz.cz(0, 1)
cx_from_cz.h(1)

print("CX == H·CZ·H:", np.allclose(Operator(cx_direct).data,
                                    Operator(cx_from_cz).data))
cx_from_cz.draw('mpl')

## 5. Statevector Simulation

`Statevector` gives the exact quantum state without measurement. Use it to inspect amplitudes, compute probabilities, and plot the Bloch sphere.

In [ ]:
# Evolve a two-qubit circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

sv = Statevector.from_label('00').evolve(qc)

print("Statevector:", np.round(sv.data, 4))
print("Probabilities:", sv.probabilities_dict())

In [ ]:
# Compare Statevector probabilities to AerSimulator shot-based counts
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

qc_meas = QuantumCircuit(2, 2)
qc_meas.h(0)
qc_meas.cx(0, 1)
qc_meas.measure([0, 1], [0, 1])

sim = AerSimulator()
t = transpile(qc_meas, sim)
counts = sim.run(t, shots=4096).result().get_counts()

# Normalize counts to probabilities for comparison
total = sum(counts.values())
sim_probs = {k: v / total for k, v in counts.items()}
exact_probs = sv.probabilities_dict()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_histogram(exact_probs, ax=axes[0], title='Exact (Statevector)')
plot_histogram(counts, ax=axes[1], title='Sampled (4096 shots)')
plt.tight_layout()
plt.show()

In [ ]:
# Statevector can also be obtained from AerSimulator using save_statevector
# This is useful for larger circuits where Statevector.evolve() may be slow

qc_sv = QuantumCircuit(2)
qc_sv.h(0)
qc_sv.cx(0, 1)
qc_sv.save_statevector()   # instruction to save the state

sim_sv = AerSimulator(method='statevector')
t = transpile(qc_sv, sim_sv)
result = sim_sv.run(t).result()
sv_from_aer = result.get_statevector()

print("Statevector from Aer:", np.round(sv_from_aer.data, 4))

## 6. Circuit Visualization Options

In [ ]:
qc = QuantumCircuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.barrier()
qc.measure([0, 1, 2], [0, 1, 2])

# Text drawing (works everywhere)
print(qc.draw('text'))

In [ ]:
# Matplotlib drawing with options
qc.draw('mpl', initial_state=True)

In [ ]:
# reverse_bits=True puts qubit 0 at the bottom (textbook convention)
qc.draw('mpl', reverse_bits=True)

In [ ]:
# fold controls line wrapping; fold=-1 disables it
# Useful for wide circuits in notebooks
long_qc = QuantumCircuit(1)
for _ in range(20):
    long_qc.h(0)
    long_qc.t(0)

print("With default fold:")
display(long_qc.draw('mpl'))

print("With fold=-1 (no wrapping):")
display(long_qc.draw('mpl', fold=-1))

## 7. Exercises

### Exercise 1: Gate sequences and state prediction

**Before running any code**, predict the output statevector for each of the following circuits starting from $|0\rangle$. Then verify your prediction with `Statevector`.

1. X then H
2. H then Z
3. H then S then H
4. T then T then T then T (four T gates)

For each circuit, also run it on `AerSimulator` with 2048 shots and compare the histogram to the exact probabilities.

In [ ]:
# Exercise 1: your solution here

circuits = {
    'X then H':       lambda qc: (qc.x(0), qc.h(0)),
    'H then Z':       lambda qc: (qc.h(0), qc.z(0)),
    'H then S then H': lambda qc: (qc.h(0), qc.s(0), qc.h(0)),
    'T^4':            lambda qc: [qc.t(0) for _ in range(4)],
}

for name, apply in circuits.items():
    qc = QuantumCircuit(1)
    apply(qc)
    sv = Statevector.from_label('0').evolve(qc)
    print(f"{name}: {np.round(sv.data, 4)}")

### Exercise 2: Building a gate from primitives

The Y gate has the matrix:
$$Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$$

1. Apply `qc.y(0)` to $|0\rangle$ and $|1\rangle$. What states do you get?
2. The Y gate can be written as $Y = iXZ$. Verify this identity using `Operator` and `np.allclose`.
3. The Y gate can also be decomposed as $Y = S X S^\dagger$. Verify this. (`qc.sdg(0)` applies the inverse of S.)

In [ ]:
# Exercise 2: your solution here

# Part 1: Y on |0⟩ and |1⟩
# TODO

# Part 2: Y = iXZ
# TODO

# Part 3: Y = S X Sdg
# TODO


### Exercise 3: Two-qubit state inspection

1. Build a circuit that prepares the state $|{-}{-}\rangle = |{-}\rangle \otimes |{-}\rangle$. (Apply H then X to both qubits, or equivalently X then H to both.)
2. Use `Statevector.probabilities_dict()` to show the four measurement probabilities. What do you expect?
3. Apply a CZ gate to the state $|{+}{+}\rangle = H|0\rangle \otimes H|0\rangle$. What is the output statevector? Show that CZ maps $|{+}{+}\rangle$ to a state with a definite phase on $|11\rangle$.

In [ ]:
# Exercise 3: your solution here

# Part 1 and 2: |--⟩ state
# TODO

# Part 3: CZ applied to |++⟩
# TODO


### Exercise 4: Custom gate library

Define a Python function `bell_prep(qc, q0, q1)` that appends the Bell state preparation circuit (H on `q0`, then CX with `q0` controlling `q1`) to any given circuit. Then:

1. Use `bell_prep` to build a 4-qubit circuit that prepares two independent Bell pairs: qubits (0, 1) and qubits (2, 3).
2. Compute the statevector. Which basis states have non-zero amplitude?
3. Run on `AerSimulator` with 2048 shots and plot the histogram.
4. How many distinct outcomes appear? Why?

In [ ]:
# Exercise 4: your solution here

def bell_prep(qc, q0, q1):
    """Append Bell state preparation to circuit qc for qubit pair (q0, q1)."""
    # TODO
    pass

# Build the 4-qubit circuit
qc = QuantumCircuit(4, 4)
# TODO: call bell_prep for both pairs
# TODO: measure all and run


## Summary

In this lesson you:

- Represented qubit states as complex unit vectors using `Statevector`.
- Learned the matrix form and Bloch sphere interpretation of X, H, S, T, and Rz gates.
- Used `Operator` to extract gate matrices and verify identities with `np.allclose`.
- Applied CX and CZ gates, confirmed their truth tables, and proved they are interconvertible via Hadamard.
- Compared exact `Statevector` probabilities to shot-based `AerSimulator` counts.
- Explored circuit visualization options: `'text'`, `'mpl'`, `reverse_bits`, `fold`, `initial_state`.

**Lesson 3** covers measurement in depth: classical registers, shot-based sampling, measurement statistics, and the first look at how hardware noise distorts ideal outcomes.